In [ ]:
from datasets import load_dataset

ds = load_dataset("opus100", "ar-en")

ar-en/test-00000-of-00001.parquet:   0%|          | 0.00/214k [00:00<?, ?B/s]

ar-en/train-00000-of-00001.parquet:   0%|          | 0.00/99.3M [00:00<?, ?B/s]

ar-en/validation-00000-of-00001.parquet:   0%|          | 0.00/979k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

In [ ]:
ds["train"][0]

{'translation': {'ar': 'و هذه؟', 'en': 'And this?'}}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#model_name = "facebook/mbart-large-50-many-to-many-mmt"
#model_name = "facebook/nllb-200-distilled-600M"
model_name="Helsinki-NLP/opus-mt-ar-en"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:

train_ds = ds["train"].select(range(2000))
test_ds = ds["test"].select(range(1000))


In [ ]:
train_ds

Dataset({
    features: ['translation'],
    num_rows: 2000
})

In [ ]:
test_ds

Dataset({
    features: ['translation'],
    num_rows: 1000
})

In [ ]:
def preprocess(example):
    src = example["translation"]["ar"]
    tgt = example["translation"]["en"]

    model_inputs = tokenizer(
        src,
        text_target=tgt,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    return model_inputs

In [ ]:
tokenized_train = train_ds.map(preprocess)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
tokenized_test = test_ds.map(preprocess)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)



In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.450519,0.154853
2,0.098144,0.155751


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.27433141708374026, metrics={'train_runtime': 7983.1721, 'train_samples_per_second': 0.501, 'train_steps_per_second': 0.125, 'total_flos': 135593459712000.0, 'train_loss': 0.27433141708374026, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.15575061738491058,
 'eval_runtime': 529.5359,
 'eval_samples_per_second': 1.888,
 'eval_steps_per_second': 0.236,
 'epoch': 2.0}

In [ ]:
device = model.device

text = "أنا أحب الموز و لكنى اكره البرتقال"

inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_length=50
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

I love bananas, but I hate oranges.


In [ ]:
trainer.save_model("./my_model")
tokenizer.save_pretrained("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model/tokenizer_config.json',
 './my_model/vocab.json',
 './my_model/source.spm',
 './my_model/target.spm',
 './my_model/added_tokens.json')

In [ ]:
!zip -r my_model.zip /content/my_model

  adding: content/my_model/ (stored 0%)
  adding: content/my_model/source.spm (deflated 55%)
  adding: content/my_model/vocab.json (deflated 77%)
  adding: content/my_model/target.spm (deflated 49%)
  adding: content/my_model/generation_config.json (deflated 43%)
  adding: content/my_model/tokenizer_config.json (deflated 67%)
  adding: content/my_model/config.json (deflated 63%)
  adding: content/my_model/training_args.bin (deflated 53%)
  adding: content/my_model/model.safetensors (deflated 7%)


In [ ]:
from google.colab import files
files.download("my_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.push_to_hub("Basma21/my-model")
tokenizer.push_to_hub("Basma21/my-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zrttf_r/model.safetensors:   0%|          |  278kB /  306MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpf787445w/target.spm : 100%|##########|  802kB /  802kB            

  /tmp/tmpf787445w/source.spm : 100%|##########|  917kB /  917kB            

CommitInfo(commit_url='https://huggingface.co/Basma21/my-model/commit/1df750b87f35221c157e7fe93fc624df28aaed05', commit_message='Upload tokenizer', commit_description='', oid='1df750b87f35221c157e7fe93fc624df28aaed05', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Basma21/my-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Basma21/my-model'), pr_revision=None, pr_num=None)